In [ ]:

import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors


In [ ]:

DATASET_PATH = "" 

df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)
df.head()


In [ ]:

drop_keywords = ["id","flow","timestamp","time"]

drop_cols = [c for c in df.columns if any(k in c.lower() for k in drop_keywords)]

df = df.drop(columns=drop_cols, errors="ignore")

print("Dropped columns:", drop_cols)


In [ ]:

possible_labels = ["label","Label","attack","Attack","class","Class","type","Type","attack_cat"]

label_col = None

for col in possible_labels:
    if col in df.columns:
        label_col = col
        break

if label_col is None:
    raise ValueError("Label column not found")

print("Detected label column:", label_col)


In [ ]:

for col in df.columns:

    if df[col].dtype == "object":
        df[col] = df[col].fillna("unknown")
    else:
        df[col] = df[col].fillna(df[col].median())


In [ ]:

encoders = {}

cat_cols = df.select_dtypes(include=["object"]).columns

for col in cat_cols:

    if col == label_col:
        continue

    le = LabelEncoder()

    df[col] = le.fit_transform(df[col].astype(str))

    encoders[col] = le

print("Encoded columns:", list(encoders.keys()))


In [ ]:
from sklearn.preprocessing import LabelEncoder

y_raw = df[label_col]

# automatically encode labels for any dataset
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

X = df.drop(columns=[label_col])

print("Classes detected:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))


In [ ]:

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Feature matrix shape:", X_scaled.shape)


In [ ]:

def generalized_pida(X, y, k=5, samples_per_point=2):

    X = np.array(X)
    y = np.array(y)
    minority_class = 1

    X_min = X[y == minority_class]

    nbrs = NearestNeighbors(n_neighbors=k).fit(X_min)

    synthetic_samples = []

    for i in range(len(X_min)):

        neighbors = nbrs.kneighbors([X_min[i]], return_distance=False)[0]

        for _ in range(samples_per_point):

            neighbor = X_min[np.random.choice(neighbors)]

            alpha = np.random.rand()

            synthetic = X_min[i] + alpha * (neighbor - X_min[i])

            noise = np.random.normal(0, 0.01, size=synthetic.shape)

            synthetic = synthetic + noise

            synthetic_samples.append(synthetic)

    synthetic_samples = np.array(synthetic_samples)

    y_syn = np.ones(len(synthetic_samples))

    X_aug = np.vstack([X, synthetic_samples])

    y_aug = np.hstack([y, y_syn])

    return X_aug, y_aug


X_aug, y_aug = generalized_pida(X_scaled, y)

print("After PIDA augmentation:", X_aug.shape)


In [ ]:
import numpy as np

def augment_sample(x):

    x1 = x.copy()
    x2 = x.copy()

    # feature masking
    mask = np.random.rand(len(x1)) < 0.1
    x1[mask] = 0

    # gaussian noise
    noise = np.random.normal(0,0.01,len(x2))
    x2 = x2 + noise

    return x1,x2

In [ ]:
import torch
import torch.nn as nn

class Encoder(nn.Module):

    def __init__(self,input_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim,128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,32)

        )

    def forward(self,x):

        return self.net(x)

In [ ]:
import torch.nn.functional as F

def contrastive_loss(z1,z2,temperature=0.07):

    z1 = F.normalize(z1,dim=1)
    z2 = F.normalize(z2,dim=1)

    batch_size = z1.shape[0]

    representations = torch.cat([z1,z2],dim=0)

    similarity_matrix = torch.matmul(
        representations,
        representations.T
    )

    labels = torch.arange(batch_size)
    labels = torch.cat([labels,labels])

    loss = 0

    for i in range(2*batch_size):

        numerator = torch.exp(
            similarity_matrix[i,labels==labels[i]]/temperature
        ).sum()

        denominator = torch.exp(
            similarity_matrix[i]/temperature
        ).sum()

        loss += -torch.log(numerator/denominator)

    return loss/(2*batch_size)

In [ ]:
import torch
import torch.optim as optim
import numpy as np

encoder = Encoder(X_aug.shape[1])

optimizer = optim.Adam(encoder.parameters(), lr=0.001)

X_tensor = torch.tensor(X_aug, dtype=torch.float32)

epochs = 10
batch_size = 256

for epoch in range(epochs):

    perm = torch.randperm(X_tensor.size(0))

    for i in range(0, len(perm), batch_size):

        idx = perm[i:i+batch_size]
        batch = X_tensor[idx]

        x1 = []
        x2 = []

        for sample in batch:

            a, b = augment_sample(sample.numpy())

            x1.append(a)
            x2.append(b)

        x1 = torch.from_numpy(np.array(x1)).float()
        x2 = torch.from_numpy(np.array(x2)).float()

        z1 = encoder(x1)
        z2 = encoder(x2)

        loss = contrastive_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch", epoch, "Loss", loss.item())

In [ ]:
with torch.no_grad():

    embeddings = encoder(
        torch.tensor(X_aug,dtype=torch.float32)
    ).numpy()

In [ ]:
import numpy as np

X_fused = np.concatenate([X_aug, embeddings], axis=1)

In [ ]:
print(X_aug.shape)
print(embeddings.shape)
print(X_fused.shape)

In [ ]:
np.save("X_fused.npy", X_fused)
np.save("y_fused.npy", y_aug)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TabularDataset(Dataset):

    def __init__(self,X,y):
        self.X = torch.tensor(X,dtype=torch.float32)
        self.y = torch.tensor(y,dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        return self.X[idx],self.y[idx]

dataset = TabularDataset(X_fused,y_aug)

loader = DataLoader(dataset,batch_size=256,shuffle=True)

In [ ]:
import torch.nn as nn

class CTVAE(nn.Module):

    def __init__(self,input_dim,latent_dim,n_classes):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim+n_classes,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(64,latent_dim)
        self.fc_logvar = nn.Linear(64,latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim+n_classes,64),
            nn.ReLU(),
            nn.Linear(64,128),
            nn.ReLU(),
            nn.Linear(128,input_dim)
        )

    def encode(self,x,y):

        xy = torch.cat([x,y],dim=1)

        h = self.encoder(xy)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu,logvar

    def reparameterize(self,mu,logvar):

        std = torch.exp(0.5*logvar)

        eps = torch.randn_like(std)

        return mu + eps*std

    def decode(self,z,y):

        zy = torch.cat([z,y],dim=1)

        return self.decoder(zy)

    def forward(self,x,y):

        mu,logvar = self.encode(x,y)

        z = self.reparameterize(mu,logvar)

        recon = self.decode(z,y)

        return recon,mu,logvar

In [ ]:
def vae_loss(recon,x,mu,logvar):

    recon_loss = ((recon-x)**2).mean()

    kl = -0.5*torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    return recon_loss + kl

In [ ]:
import torch.optim as optim
import torch.nn.functional as F

n_classes = len(np.unique(y_aug))

model = CTVAE(
    input_dim=X_fused.shape[1],
    latent_dim=16,
    n_classes=n_classes
)

optimizer = optim.Adam(model.parameters(),lr=0.001)

epochs = 50

for epoch in range(epochs):

    for x,y in loader:

        y_onehot = F.one_hot(y,n_classes).float()

        recon,mu,logvar = model(x,y_onehot)

        loss = vae_loss(recon,x,mu,logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch:",epoch,"Loss:",loss.item())

In [ ]:
def generate_samples(model,class_id,n_samples):

    z = torch.randn(n_samples,16)

    y = torch.full((n_samples,),class_id)

    y_onehot = F.one_hot(y,n_classes).float()

    samples = model.decode(z,y_onehot)

    return samples.detach().numpy()

In [ ]:
X_syn = generate_samples(model,class_id=9,n_samples=8000)

y_syn = np.full(len(X_syn),9)

X_final = np.vstack([X_fused,X_syn])
y_final = np.hstack([y_aug,y_syn])

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=0.2,
    stratify=y_final,
    random_state=42
)

In [ ]:
model = xgb.XGBClassifier(

    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    objective='multi:softprob',
    eval_metric='mlogloss',

    tree_method='hist'
)

In [ ]:
model.fit(
    X_train,
    y_train,
    eval_set=[(X_test,y_test)],
    verbose=True
)

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)

print("Accuracy:",accuracy)
print(classification_report(y_test,y_pred))

In [ ]:
import pandas as pd

# Save the final dataset for sharing
final_df = pd.DataFrame(X_final)
final_df['label'] = y_final
final_df.to_csv('final_shared_dataset.csv', index=False)
print("Final dataset saved as final_shared_dataset.csv")